# canteen_checkout on Colab

One-time setup:
1. Upload `canteen_checkout.zip` and `unimib_yolo.zip` to `MyDrive/canteen/` in Google Drive
2. Menu **Runtime → Change runtime type → T4 GPU**

Data is unpacked to Colab's local disk `/content/work` (fast reads); all outputs (weights, gallery, reports) go straight to `MyDrive/canteen/runs/`, so nothing is lost if the runtime disconnects.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob, json
DRIVE = '/content/drive/MyDrive/canteen'
RUNS  = f'{DRIVE}/runs'
WORK  = '/content/work'
os.makedirs(RUNS, exist_ok=True)
os.makedirs(WORK, exist_ok=True)

# same layout as on the Mac: canteen_checkout/ next to unimib_yolo/
for name in ('canteen_checkout', 'unimib_yolo'):
    if not os.path.isdir(f'{WORK}/{name}'):
        !cp "{DRIVE}/{name}.zip" /content/ && unzip -q "/content/{name}.zip" -d "{WORK}" && rm "/content/{name}.zip"
DATA = f'{WORK}/unimib_yolo'
%cd {WORK}/canteen_checkout
!ls {DATA}

In [ ]:
# Colab ships torch/torchvision/numpy/opencv/scipy; do not reinstall them
!pip install -q ultralytics timm faiss-cpu ensemble-boxes
!python tests/smoke_test.py 2>&1 | tail -4

## 1. Train the detector

Colab has only 2 CPUs, hence `--workers 2`. `yolo11s-seg` with batch 16 fits a T4; for a quick baseline first, set `MODEL` to `yolo11n-seg.pt`.

In [ ]:
MODEL = 'yolo11s-seg.pt'
!python train_detector.py --data {DATA}/data.yaml --model {MODEL} \
    --device 0 --batch 16 --workers 2 --epochs 100 --imgsz 640 \
    --project {RUNS}/detector --name unimib

**Resume after a disconnect**: re-run the setup cells above (the first 3), then run the cell below.

In [ ]:
last = max(glob.glob(f'{RUNS}/detector/unimib*/weights/last.pt'), key=os.path.getmtime)
print('resume from', last)
!python train_detector.py --data {DATA}/data.yaml --model "{last}" --resume --device 0 --workers 2

## 2. Build the gallery (pretrained DINOv2 ViT-S/14)

In [ ]:
!python build_gallery.py --crops {DATA}/crops/train --out {RUNS}/gallery.npz --device cuda

## 3. Calibrate thresholds on val (GT boxes), then evaluate end to end on test

Tune thresholds on val only; test is for reporting.

In [ ]:
!python evaluate.py --dataset {DATA} --gallery {RUNS}/gallery.npz --detector oracle \
    --split val --calibrate 0.98 --device cuda --out {RUNS}/report_val_calib.json > /dev/null
cal = json.load(open(f'{RUNS}/report_val_calib.json'))['calibration']['suggested']
print(cal)
ACCEPT_SIM, MARGIN = cal['accept_sim'], cal['margin']

In [ ]:
BEST = max(glob.glob(f'{RUNS}/detector/unimib*/weights/best.pt'), key=os.path.getmtime)
print('detector:', BEST)
!python evaluate.py --dataset {DATA} --gallery {RUNS}/gallery.npz --detector "{BEST}" \
    --split test --accept-sim {ACCEPT_SIM} --margin {MARGIN} --device cuda \
    --out {RUNS}/report_test.json --save-vis {RUNS}/vis_test > /dev/null

def summary(path):
    r = json.load(open(path))
    print(json.dumps({'detection': r['detection'],
                      'top1': r['recognition_on_gt_crops']['top1'],
                      'precision_of_accepted': r['recognition_on_gt_crops']['precision_of_accepted'],
                      'end_to_end': r['end_to_end'],
                      'thresholds': r['thresholds']}, indent=2, ensure_ascii=False))
summary(f'{RUNS}/report_test.json')

## 4. (Optional) ArcFace fine-tune of the embedder, rebuild the gallery, recalibrate and evaluate

In [ ]:
!python finetune_embedder.py --train {DATA}/crops/train --val {DATA}/crops/val \
    --out {RUNS}/embedder --balanced --device cuda --workers 2

In [ ]:
CKPT = f'{RUNS}/embedder/best.pt'
!python build_gallery.py --crops {DATA}/crops/train --out {RUNS}/gallery_ft.npz --checkpoint "{CKPT}" --device cuda
!python evaluate.py --dataset {DATA} --gallery {RUNS}/gallery_ft.npz --detector oracle \
    --split val --calibrate 0.98 --device cuda --out {RUNS}/report_val_calib_ft.json > /dev/null
cal = json.load(open(f'{RUNS}/report_val_calib_ft.json'))['calibration']['suggested']
print(cal)
!python evaluate.py --dataset {DATA} --gallery {RUNS}/gallery_ft.npz --detector "{BEST}" \
    --split test --accept-sim {cal['accept_sim']} --margin {cal['margin']} --device cuda \
    --out {RUNS}/report_test_ft.json --save-vis {RUNS}/vis_test_ft > /dev/null
summary(f'{RUNS}/report_test_ft.json')

## 5. Test new images with the trained models

Steps 1–3 must be done first (after a disconnect, re-running the first 3 setup cells is enough: weights and gallery are on Drive).
To use the fine-tuned embedder from step 4, set `USE_FT = True`. A price list is optional: put a filled-in `prices.csv` in `MyDrive/canteen/` (format as in `unimib_yolo/prices_template.csv`); it runs without one too.

In [ ]:
import yaml
from IPython.display import Image, display

USE_FT = False
BEST = max(glob.glob(f'{RUNS}/detector/unimib*/weights/best.pt'), key=os.path.getmtime)
tag = '_ft' if USE_FT else ''
cal = json.load(open(f'{RUNS}/report_val_calib{tag}.json'))['calibration']['suggested']

cfg = yaml.safe_load(open('configs/checkout.yaml'))
cfg['detector']['weights'] = BEST
cfg['index']['path'] = f'{RUNS}/gallery{tag}.npz'
cfg['embedder']['checkpoint'] = f'{RUNS}/embedder/best.pt' if USE_FT else None
cfg['decision'].update(accept_sim=cal['accept_sim'], margin=cal['margin'])
cfg['pricing']['csv'] = f'{DRIVE}/prices.csv'
CFG = f'{RUNS}/checkout_colab{tag}.yaml'
yaml.safe_dump(cfg, open(CFG, 'w'), allow_unicode=True, sort_keys=False)
print(open(CFG).read())

In [ ]:
# Put the photos in MyDrive/canteen/new_images/ (drag them in with Drive on the web)
# files.upload() is not used: with larger images or Safari it fails with "Maximum call stack size exceeded"
NEW = f'{DRIVE}/new_images'
os.makedirs(NEW, exist_ok=True)
imgs = sorted(p for p in glob.glob(f'{NEW}/*') if p.lower().endswith(('.jpg', '.jpeg', '.png')))
print(len(imgs), 'images:', [os.path.basename(p) for p in imgs])
if not imgs:
    print('files in new_images/:', os.listdir(NEW))
    print('images elsewhere on Drive:', glob.glob(f'{DRIVE}/**/*.*', recursive=True)[:20])
    raise SystemExit('No .jpg/.png found: check the folder and file extensions; files just dropped into Drive may take a moment to appear, or remount')

# WEIGHT = 540   # optional: tray weight in grams; add --weight {WEIGHT} to enable the weight cross-check
!python demo.py --config "{CFG}" --image "{NEW}" --save {RUNS}/demo
for p in imgs:
    display(Image(f'{RUNS}/demo/{os.path.basename(p)}', width=800))

## 6. Add a new dish / more photos of an existing class (no retraining)

Recognition is kNN retrieval, so adding a class = appending the new photos' embeddings to the gallery. Put the photos in `MyDrive/canteen/new_dishes/<class_name>/`;
the folder name is the class name. An existing class (e.g. `banana`) with the same folder name gets extra photos; a new name creates a new class.
The result is written to a new file `gallery_plus.npz`; the original gallery is kept for comparison.

In [ ]:
NEW_DISHES = f'{DRIVE}/new_dishes'
BEST = max(glob.glob(f'{RUNS}/detector/unimib*/weights/best.pt'), key=os.path.getmtime)
for d in sorted(os.listdir(NEW_DISHES)):
    print(d, len(os.listdir(f'{NEW_DISHES}/{d}')), 'files')

# --detector: crop with the trained detector (largest detection kept), the same way as at runtime
# if the gallery is the fine-tuned one, use --append {RUNS}/gallery_ft.npz and add --checkpoint {RUNS}/embedder/best.pt
!python build_gallery.py --append {RUNS}/gallery.npz --crops {NEW_DISHES} \
    --out {RUNS}/gallery_plus.npz --detector "{BEST}" --device cuda

In [ ]:
# regenerate the section 5 config with the new gallery, then re-run the last cell of section 5
cfg = yaml.safe_load(open(CFG))
cfg['index']['path'] = f'{RUNS}/gallery_plus.npz'
CFG = f'{RUNS}/checkout_colab_plus.yaml'
yaml.safe_dump(cfg, open(CFG, 'w'), allow_unicode=True, sort_keys=False)
print('now using', CFG)

## 7. Switch a gallery built with Italian class names to English (no re-embedding)

Galleries built before the switch to English class names (`banane`, `pane`, ...) can be relabelled in place of a rebuild. Section 6 folder names must use the English names afterwards (`new_dishes/banana/`).

In [ ]:
for name in ('gallery', 'gallery_ft', 'gallery_plus'):
    if os.path.exists(f'{RUNS}/{name}.npz'):
        !python tools/relabel_gallery.py --gallery {RUNS}/{name}.npz --out {RUNS}/{name}.npz